# IRWA Final Project — Part 2: Indexing and Evaluation

**Input:** `data/processed_corpus.pkl` (output of Part 1) and `data/validation_labels.csv`.

Structure of this notebook:
- **PART 1: Indexing**
  0. Setup and data loading
  1. Inverted index
  2. Test queries
  3. TF-IDF ranking
- **PART 2: Evaluation**
  1. Evaluation metrics
  2. Evaluation on `validation_labels.csv`
  3. Expert judgments for the test queries


# PART 1: Indexing

## 0. Setup and data loading


In [1]:
import html
import math
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Repository root: the folder that contains myapp/ and data/
ROOT = Path.cwd()
while not (ROOT / "myapp").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA = ROOT / "data"
PART2 = ROOT / "project_progress" / "part_2"

corpus = pd.read_pickle(DATA / "processed_corpus.pkl")
labels = pd.read_csv(DATA / "validation_labels.csv")

n_products = len(corpus)
print(n_products, "products loaded")

by_query = labels.groupby("query_id")["labels"]
summary = by_query.agg(["size", "sum"])
summary = summary.rename(columns={"size": "labelled", "sum": "relevant"})
print(summary)


28080 products loaded
          labelled  relevant
query_id                    
1               20        13
2               20        10


In [2]:
# Same pre-processing as the documents in Part 1, so a query uses the same terms.
try:
    stopword_list = stopwords.words("english")
    STOP_WORDS = set(stopword_list)
except LookupError:
    import nltk
    nltk.download("stopwords", quiet=True)
    stopword_list = stopwords.words("english")
    STOP_WORDS = set(stopword_list)

STEMMER = SnowballStemmer("english")

# A token is a word, or a run of punctuation so we can drop punctuation in its own step.
_TOKEN_RE = re.compile(r"[a-z0-9]+|[^a-z0-9\s]+")
_WORD_RE = re.compile(r"[a-z0-9]+")
# Hyphen or apostrophe inside a word: "t-shirt" -> "tshirt"
_INNER_JOIN_RE = re.compile(r"(?<=[a-z0-9])[-'’](?=[a-z0-9])")


def normalize_text(text):
    """Lower-case, remove accents and join hyphens inside a word."""
    if not text:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKD", text)

    chars = []
    for ch in text:
        is_accent = unicodedata.combining(ch)
        if not is_accent:
            chars.append(ch)
    text = "".join(chars)

    text = text.lower()
    text = text.replace("\xa0", " ")
    text = _INNER_JOIN_RE.sub("", text)
    return text


def tokenize(text):
    """Split the text. Punctuation stays as its own tokens."""
    return _TOKEN_RE.findall(text)


def remove_punctuation(tokens):
    """Keep only tokens made of letters and digits."""
    words = []
    for tok in tokens:
        if _WORD_RE.fullmatch(tok):
            words.append(tok)
    return words


def remove_stop_words(tokens):
    """Drop English stop words."""
    kept = []
    for tok in tokens:
        if tok not in STOP_WORDS:
            kept.append(tok)
    return kept


def stem_tokens(tokens):
    """Apply the Snowball stemmer to every token."""
    stemmed = []
    for tok in tokens:
        stem = STEMMER.stem(tok)
        stemmed.append(stem)
    return stemmed


def build_terms(text):
    """Tokenize, drop punctuation and stop words, then stem."""
    normalised = normalize_text(text)
    tokens = tokenize(normalised)
    tokens = remove_punctuation(tokens)
    tokens = remove_stop_words(tokens)

    kept = []
    for tok in tokens:
        if tok.isdigit():
            continue
        if len(tok) < 2:
            continue
        kept.append(tok)

    return stem_tokens(kept)


terms_q1 = build_terms("women full sleeve sweatshirt cotton")
terms_q2 = build_terms("men slim jeans blue")
print(terms_q1)
print(terms_q2)


['women', 'full', 'sleev', 'sweatshirt', 'cotton']
['men', 'slim', 'jean', 'blue']


## 1. Inverted index

In [3]:
def term_counts(terms):
    """How many times each term appears."""
    counts = {}
    for term in terms:
        if term not in counts:
            counts[term] = 0
        counts[term] = counts[term] + 1
    return counts


def l2_norm(counts):
    """Length of a count vector, used to normalise tf."""
    total = 0.0
    for count in counts.values():
        total = total + count * count
    return math.sqrt(total)


def create_index(corpus):
    """Inverted index, normalised tf and idf, built from all_terms."""
    index = {}
    tf = {}

    pids = corpus["pid"]
    documents = corpus["all_terms"]

    for pid, terms in zip(pids, documents):
        counts = term_counts(terms)
        if len(counts) == 0:
            continue

        norm = l2_norm(counts)
        for term, count in counts.items():
            if term not in index:
                index[term] = []
                tf[term] = {}
            index[term].append(pid)
            # tf = count / length of the document vector
            tf[term][pid] = count / norm

    n_docs = len(corpus)
    df = {}
    idf = {}
    for term, posting in index.items():
        df[term] = len(posting)
        idf[term] = math.log(n_docs / df[term])

    return index, tf, df, idf


index, tf, df, idf = create_index(corpus)
n_terms = len(index)
print("Vocabulary:", n_terms, "terms")

example = index["sweatshirt"]
print("index['sweatshirt'][:5] ->", example[:5], "... (", df["sweatshirt"], "documents)")


Vocabulary: 6304 terms
index['sweatshirt'][:5] -> ['SWSFYD33P2XCBWFG', 'SWSFYD3KMGTBZEJZ', 'SWSFYD38Y2MAT9FQ', 'SWSFVEV2YFEJKUCK', 'SWSFVEV2BWZJDJ46'] ... ( 1418 documents)


In [4]:
def conjunctive_search(query_terms, index):
    """Pids that contain every query term. One missing term means no results."""
    if len(query_terms) == 0:
        return set()

    unique_terms = []
    for term in query_terms:
        if term not in unique_terms:
            unique_terms.append(term)

    postings = []
    for term in unique_terms:
        if term not in index:
            return set()
        postings.append(index[term])

    # The shortest list first makes the intersection smaller at each step.
    postings.sort(key=len)

    docs = set(postings[0])
    n_lists = len(postings)
    for i in range(1, n_lists):
        next_docs = set(postings[i])
        docs = docs.intersection(next_docs)
        if len(docs) == 0:
            break
    return docs


demo_queries = [
    "women full sleeve sweatshirt cotton",
    "men slim jeans blue",
]
for query in demo_queries:
    terms = build_terms(query)
    docs = conjunctive_search(terms, index)
    n_docs_q = len(docs)
    print(query, "->", n_docs_q, "documents")


women full sleeve sweatshirt cotton -> 477 documents
men slim jeans blue -> 221 documents


## 2. Test queries

In [5]:
df_table = pd.DataFrame({"df": df, "idf": idf})
df_table = df_table.sort_values("df", ascending=False)
top_terms = df_table.head(30)
shown = top_terms.T
shown.round(2)


,accessori,cloth,wear,cotton,western,sleev,topwear,neck,men,women,...,comfort,full,blue,black,fabric,slim,pure,design,look,short
df,27189.00,27121.00,20968.00,19836.00,19104.00,17908.00,15310.00,14366.00,14142.00,13702.00,...,7223.00,7170.00,7139.00,5738.00,5677.0,5639.00,5522.00,5375.00,5197.00,4947.00
idf,0.03,0.03,0.29,0.35,0.39,0.45,0.61,0.67,0.69,0.72,...,1.36,1.37,1.37,1.59,1.6,1.61,1.63,1.65,1.69,1.74


In [6]:
TEST_QUERIES = {
    3: "printed hoodie women",
    4: "running shoes",
    5: "denim jacket women",
    6: "track pants men black",
    7: "slim fit formal shirt men",
}

rows = []
for qid, query in TEST_QUERIES.items():
    terms = build_terms(query)
    term_df = []
    for term in terms:
        term_df.append(df.get(term, 0))
    docs = conjunctive_search(terms, index)
    rows.append({
        "query_id": qid,
        "query": query,
        "terms": terms,
        "df of each term": term_df,
        "AND results": len(docs),
    })

query_table = pd.DataFrame(rows)
query_table = query_table.set_index("query_id")
query_table


,query,terms,df of each term,AND results
query_id,,,,
3,printed hoodie women,"[print, hoodi, women]","[9495, 414, 13702]",127
4,running shoes,"[run, shoe]","[772, 526]",80
5,denim jacket women,"[denim, jacket, women]","[1776, 1093, 13702]",82
6,track pants men black,"[track, pant, men, black]","[1807, 2012, 14142, 5738]",273
7,slim fit formal shirt men,"[slim, fit, formal, shirt, men]","[5639, 10461, 1842, 10382, 14142]",120


## 3. TF-IDF ranking

In [7]:
def rank_documents(query_terms, docs, tf, idf):
    """Cosine similarity between the TF-IDF vectors of the query and each document."""
    counts = term_counts(query_terms)
    q_norm = l2_norm(counts)

    # Weight of each query term: normalised tf * idf
    query_weight = {}
    for term, count in counts.items():
        tf_query = count / q_norm
        query_weight[term] = tf_query * idf[term]

    scored = []
    for pid in docs:
        # Dot product, only over the query terms (the other coordinates are 0)
        score = 0.0
        for term, weight in query_weight.items():
            tf_doc = tf[term][pid]
            doc_weight = tf_doc * idf[term]
            score = score + doc_weight * weight
        scored.append((pid, score))

    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored


def search_tf_idf(query, index, tf, idf):
    """Pre-process the query, keep documents with every term, then rank them."""
    terms = build_terms(query)
    docs = conjunctive_search(terms, index)
    if len(terms) == 0 or len(docs) == 0:
        return []
    return rank_documents(terms, docs, tf, idf)


def show_results(query, top=10):
    """Print how many documents matched and return the first ones."""
    results = search_tf_idf(query, index, tf, idf)
    n_results = len(results)
    print(query, ":", n_results, "results")

    n_shown = len(results)
    if n_shown > top:
        n_shown = top

    pids = []
    scores = []
    for i in range(n_shown):
        pid = results[i][0]
        score = results[i][1]
        pids.append(pid)
        score_shown = round(score, 4)
        scores.append(score_shown)

    columns = ["title", "brand_full", "selling_price", "average_rating"]
    out = corpus.loc[pids, columns]
    out = out.copy()
    out.insert(0, "score", scores)
    return out


In [8]:
VALIDATION_QUERIES = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue",
}
show_results(VALIDATION_QUERIES[1])


women full sleeve sweatshirt cotton : 477 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
SWSFMTNHEM5SNHVZ,2.8094,Full Sleeve Solid Women Sweatshirt,arbour,854.0,4.2
SWSFMJGS8HVBPEH6,2.4634,Full Sleeve Solid Women Sweatshirt,arbour,759.0,4.3
SWSFMJGTGZMJXK5H,2.4473,Full Sleeve Solid Women Sweatshirt,arbour,949.0,4.2
SWSFMJGTVFW3NAXZ,2.4473,Full Sleeve Solid Women Sweatshirt,arbour,759.0,4.2
SWSFMJGTQFMZGZQH,2.4473,Full Sleeve Solid Women Sweatshirt,arbour,759.0,4.2
SWSFZVTTQCB4SJ7F,2.3169,Full Sleeve Solid Women Sweatshirt,levis,2239.0,NaN
SWSFZVTNGM4HG8BC,2.3169,Full Sleeve Printed Women Sweatshirt,levis,1042.0,4.2
SWSFWTND3EPMFCQD,2.2293,Full Sleeve Solid Women Sweatshirt,viralee retail,647.0,4.7
SWSFWTNDJFCF72WU,2.2192,Full Sleeve Solid Women Sweatshirt,viralee retail,647.0,4.7


In [9]:
show_results(VALIDATION_QUERIES[2])

men slim jeans blue : 221 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
JEAF2QF6HNZHDXVX,2.1486,Slim Men Dark Blue Jeans,absolute,856.0,3.8
JEAFJM2P2FQQJF3C,2.1216,Slim Men Dark Blue Jeans,absolute,589.0,4.1
JEAFRAQXEKGUPNUN,2.0999,Slim Men Blue Jeans,levis,3999.0,NaN
JEAFSKYHTE76YWH7,1.9840,Slim Men Blue Jeans,ecko unltd,974.0,2.5
JEAFSKYHDYZK5SHZ,1.9840,Slim Men Blue Jeans,ecko unltd,899.0,2.5
JEAFSKYHRVZSABPR,1.9840,Slim Men Blue Jeans,ecko unltd,974.0,3.3
JEAFDZRVABZZ2PQ3,1.9830,Stretchable Slim Men Blue Jeans (Pack of 2),nebraska,1275.0,3.9
JEAFWZXTFGMP9GCN,1.9437,Slim Men Blue Jeans,levis,2305.0,2.2
JEAFEC2GEMGBWHA5,1.9368,Slim Men Light Blue Jeans,nebraska,629.0,3.9


In [10]:
show_results(TEST_QUERIES[3])

printed hoodie women : 127 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
SWSFFVKBHZVGESZR,4.4264,Full Sleeve Graphic Print Men Sweatshirt,cupidstore,999.0,4.3
SWSFFVKBAZGKCQCX,4.4062,Full Sleeve Graphic Print Women Sweatshirt,cupidstore,999.0,4.3
SWSFFUFZMAHCJ5KG,4.3819,Full Sleeve Graphic Print Men Sweatshirt,cupidstore,930.0,4.3
SWSFFUFZW8ZH5UAJ,4.3819,Full Sleeve Graphic Print Men Sweatshirt,cupidstore,930.0,4.3
SWSFFUFZ6PVYHSGG,4.3674,Full Sleeve Graphic Print Men Sweatshirt,cupidstore,949.0,4.3
SWSFFVKBF5U5AHBR,4.0996,Full Sleeve Printed Women Sweatshirt,cupidstore,999.0,3.2
SWSFFVKBB8FHEB7R,4.0883,Full Sleeve Printed Women Sweatshirt,cupidstore,999.0,3.2
SWSFFVKBYG9ZUBTW,4.0883,Full Sleeve Printed Women Sweatshirt,cupidstore,999.0,3.2
SWSFFVKBQUWH72KH,4.0829,Full Sleeve Printed Women Sweatshirt,cupidstore,999.0,3.2


In [11]:
show_results(TEST_QUERIES[4])

running shoes : 80 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
SHOFC585JH9RGXWG,8.8916,Cricket Shoes For Women (White),vector,1299.0,4.2
SOCFVYHYZFJ4FFNG,8.2766,"Men Solid Low Cut, Peds/Footie/No-Show (Pack ...",aeloader,249.0,4.2
SHOFWFVWRSNK5AZJ,7.4416,Running Shoes For Women (Multicolor),bonexy,474.0,3.9
SHOFUH9JBHVZC5TU,7.3218,Sneakers For Women (Red),bonexy,404.0,3.8
SHOFN3EJZVGC6GH8,7.3218,RS-7250 Running Shoes For Women (Navy),vector,990.0,3.9
SHOFG5TXSM7M57MY,7.1420,Running Shoes For Women (Grey),bonexy,474.0,3.2
SHOFTZFQP8W9G86F,7.1155,Running Shoes For Women (Black),bonexy,341.0,4.1
SHOFTZFQQKDWFYE6,7.1155,Running Shoes For Women (Orange),bonexy,341.0,4.1
SHOF98H4X8PU3EGW,7.0994,Running Shoes For Women (Blue),bonexy,474.0,3.2


In [12]:
show_results(TEST_QUERIES[5])

denim jacket women : 82 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
JCKFXAKEY2ANHJGW,4.7857,Full Sleeve Solid Women Denim Jacket,voxati,1199.0,3.7
JCKFWC5YW4Z3YYZX,4.7857,Full Sleeve Solid Women Denim Jacket,voxati,1199.0,4.0
JCKFWC5ZQ8QVSETD,4.7772,Full Sleeve Solid Women Denim Jacket,voxati,1199.0,3.7
JCKFWC5ZKPYQ6ZGH,4.7772,Full Sleeve Solid Women Denim Jacket,voxati,1199.0,3.7
JCKFWC5YGQENYT6B,4.7450,Full Sleeve Solid Women Denim Jacket,voxati,1199.0,4.0
JCKFMHS8ANSPXK72,4.4013,Full Sleeve Printed Women Jacket,cherry,1036.0,4.4
JCKFZ33GHPEXHFUZ,4.3712,Full Sleeve Solid Women Jacket,,1149.0,NaN
JCKFPEVF2XP9VH2E,4.3056,Full Sleeve Washed Women Denim Jacket,solid styles,1424.0,3.7
JCKF7KY3NQUGDHE2,4.2162,Full Sleeve Solid Women Denim Casual Jacket,voxati,791.0,1.0


In [13]:
show_results(TEST_QUERIES[6])

track pants men black : 273 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
TKPFZMQ8YMWNGXEU,4.1895,Solid Men Black Track Pants,linsden,399.0,NaN
TKPF3FPNZ5XCHZ6R,4.0196,"Solid Men Black, Grey Track Pants",fairdeals,538.0,2.6
TKPF3FPJNVVBKPRG,4.0196,"Solid Men Blue, Black Track Pants",fairdeals,631.0,2.6
TKPF3FPKHNWZ3UMG,4.0196,"Solid Men Blue, Black Track Pants",fairdeals,538.0,2.6
TKPF3FPZRRJUWTAZ,4.0196,"Solid Men Blue, Black Track Pants",fairdeals,538.0,2.6
TKPF3FPKTYM55SAV,4.0196,"Solid Men Black, Grey Track Pants",fairdeals,538.0,2.6
TKPF3FPQGCHXHGRM,4.0196,"Solid Men Black, Grey Track Pants",fairdeals,538.0,2.6
TKPEYQ6YNHW8FV5R,3.9277,"Solid Men Black, Orange Track Pants",fairdeals,375.0,3.5
TKPFZHV4MZB4TF7G,3.7647,"Striped Men Grey, Black Track Pants",glitchy,499.0,NaN


In [14]:
show_results(TEST_QUERIES[7])

slim fit formal shirt men : 120 results


,score,title,brand_full,selling_price,average_rating
pid,,,,,
SHTFDYRQRYGTW4PX,1.7056,Men Slim Fit Printed Button Down Collar Formal...,mods,799.0,3.7
SHTFDYPHFAB6UKF8,1.6942,Men Slim Fit Printed Button Down Collar Formal...,mods,720.0,3.7
SHTFDYSQZ9AG7ZTV,1.6721,Men Slim Fit Checkered Button Down Collar Form...,mods,799.0,3.9
TROFZEYCMFEZZ7PP,1.4902,Slim Fit Men Brown Pure Cotton Trousers,ksx,735.0,2.5
TROFF9G3ZA4GV9DM,1.4673,Slim Fit Men Beige Viscose Trousers,adam parker,1299.0,4.1
SHTFW52ZURHFKNTZ,1.4388,Men Slim Fit Printed Button Down Collar Formal...,truemoda,758.0,4.2
SHTFW53YYCHYUHZP,1.4388,Men Slim Fit Solid Spread Collar Formal Shirt,truemoda,853.0,4.2
SHTFYVTNUEGD79EF,1.4034,Men Slim Fit Printed Formal Shirt,purple state,629.0,2.9
SHTFYVTZEWGW8AVC,1.4034,Men Slim Fit Solid Formal Shirt,purple state,629.0,2.9


# PART 2: Evaluation

## 1. Evaluation metrics


In [15]:
def top_labels(y_true, y_score, k):
    """Relevance labels of the k documents with the highest score."""
    y_true = np.asarray(y_true)
    order = np.argsort(y_score)
    order = order[::-1]
    n = len(order)
    if n > k:
        n = k
    order = order[:n]
    return np.take(y_true, order)


def precision_at_k(y_true, y_score, k=10):
    """Share of the top K documents that are relevant."""
    top = top_labels(y_true, y_score, k)
    relevant = np.sum(top == 1)
    return float(relevant) / k


def recall_at_k(y_true, y_score, k=10, n_relevant=None):
    """Share of all relevant documents that appear in the top K."""
    y_true = np.asarray(y_true)
    if n_relevant is None:
        total = y_true.sum()
    else:
        total = n_relevant
    total = float(total)
    if total == 0:
        return 0.0
    top = top_labels(y_true, y_score, k)
    found = np.sum(top == 1)
    return float(found) / total


def avg_precision_at_k(y_true, y_score, k=10):
    """Mean of the precision at the rank of each relevant document in the top K."""
    y_true = np.asarray(y_true)
    order = np.argsort(y_score)
    order = order[::-1]

    n = len(order)
    if n > k:
        n = k

    precisions = []
    n_rel = 0
    for i in range(n):
        label = y_true[order[i]]
        if label == 1:
            n_rel = n_rel + 1
            precision_i = n_rel / (i + 1)
            precisions.append(precision_i)

    if n_rel == 0:
        return 0.0
    mean_precision = sum(precisions) / n_rel
    return float(mean_precision)


def f1_at_k(y_true, y_score, k=10, n_relevant=None):
    """Harmonic mean of Precision@K and Recall@K."""
    p = precision_at_k(y_true, y_score, k)
    r = recall_at_k(y_true, y_score, k, n_relevant)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)


def map_at_k(per_query, k=10):
    """Mean Average Precision@K over the queries."""
    values = []
    for y_true, y_score in per_query.values():
        ap = avg_precision_at_k(y_true, y_score, k)
        values.append(ap)
    mean_ap = np.mean(values)
    return float(mean_ap), values


def rr_at_k(y_true, y_score, k=10):
    """1 / rank of the first relevant document in the top K, or 0."""
    top = top_labels(y_true, y_score, k)
    n_rel = np.sum(top)
    if n_rel == 0:
        return 0.0
    is_relevant = top == 1
    position = np.argmax(is_relevant)
    return 1.0 / (position + 1)


def mrr_at_k(per_query, k=10):
    """Mean Reciprocal Rank@K over the queries."""
    values = []
    for y_true, y_score in per_query.values():
        rr = rr_at_k(y_true, y_score, k)
        values.append(rr)
    mean_rr = np.mean(values)
    return float(mean_rr)


def dcg_at_k(y_true, y_score, k=10):
    """DCG@K. The gain is 2^rel - 1 and the discount is log2(rank + 1)."""
    top = top_labels(y_true, y_score, k)
    dcg = 0.0
    for i, rel in enumerate(top):
        gain = (2 ** rel) - 1
        discount = math.log2(i + 2)
        dcg = dcg + gain / discount
    return float(dcg)


def ndcg_at_k(y_true, y_score, k=10, ideal_true=None):
    """DCG of our ranking divided by the DCG of the ideal ranking."""
    if ideal_true is None:
        ideal = np.asarray(y_true)
    else:
        ideal = np.asarray(ideal_true)
    dcg_max = dcg_at_k(ideal, ideal, k)
    if dcg_max == 0:
        return 0.0
    dcg = dcg_at_k(y_true, y_score, k)
    return dcg / dcg_max


In [16]:
# Small examples, checked by hand
y_true = np.array([1, 1, 0, 1, 0, 0, 1])
y_scores = np.array([7, 6, 5, 4, 3, 2, 1])

p_at_3 = precision_at_k(y_true, y_scores, 3)          # 2/3
r_at_3 = recall_at_k(y_true, y_scores, 3)             # 2/4
ap_at_4 = avg_precision_at_k(y_true, y_scores, 4)     # (1 + 1 + 3/4) / 3
print("P@3 ", p_at_3)
print("R@3 ", r_at_3)
print("AP@4", ap_at_4)

labels_rr = np.array([0, 1, 0, 1, 1])
scores_rr = np.array([0.9, 0.5, 0.6, 0.7, 0.2])
rr = rr_at_k(labels_rr, scores_rr, 5)                 # first relevant at rank 2
print("RR@5", rr)

perfect_scores = np.array([4, 3, 0, 2, 0, 0, 1])
ndcg = ndcg_at_k(y_true, perfect_scores, 7)           # already in the ideal order
print("NDCG@7 (perfect order)", ndcg)


P@3  0.6666666666666666
R@3  0.5
AP@4 0.9166666666666666


RR@5 0.5
NDCG@7 (perfect order) 1.0


In [17]:
def judged_ranking(query, judgments):
    """Scores of the judged documents that contain every query term."""
    ranked = search_tf_idf(query, index, tf, idf)
    scores = {}
    for pid, score in ranked:
        scores[pid] = score

    # A judged document that misses a query term is not retrieved.
    # It stays in all_labels, so Recall and the ideal NDCG still count it.
    mask = judgments["pid"].isin(scores)
    retrieved = judgments[mask]

    y_true = retrieved["label"].to_numpy()
    y_score = retrieved["pid"].map(scores)
    y_score = y_score.to_numpy()
    all_labels = judgments["label"].to_numpy()
    return y_true, y_score, all_labels


def evaluate(queries, judgments, ks=(5, 10)):
    """Per-query metrics, plus MAP, MRR and mean NDCG. Rounded to 3 decimals."""
    columns = {}
    per_query = {}

    for qid, query in queries.items():
        mask = judgments["query_id"] == qid
        subset = judgments[mask]
        y_true, y_score, all_true = judged_ranking(query, subset)
        per_query[qid] = (y_true, y_score)

        n_rel = all_true.sum()
        n_rel = int(n_rel)
        col = {}
        for k in ks:
            col[f"P@{k}"] = precision_at_k(y_true, y_score, k)
            col[f"R@{k}"] = recall_at_k(y_true, y_score, k, n_rel)
            col[f"AP@{k}"] = avg_precision_at_k(y_true, y_score, k)
            col[f"F1@{k}"] = f1_at_k(y_true, y_score, k, n_rel)
            col[f"RR@{k}"] = rr_at_k(y_true, y_score, k)
            col[f"NDCG@{k}"] = ndcg_at_k(y_true, y_score, k, all_true)
        columns[f"query {qid}"] = col

    per_query_table = pd.DataFrame(columns)

    overall = {}
    for k in ks:
        mean_ap, _ = map_at_k(per_query, k)
        mean_ndcg = per_query_table.loc[f"NDCG@{k}"]
        mean_ndcg = mean_ndcg.mean()
        overall[f"MAP@{k}"] = mean_ap
        overall[f"MRR@{k}"] = mrr_at_k(per_query, k)
        overall[f"mean NDCG@{k}"] = mean_ndcg

    overall_table = pd.DataFrame({"all queries": overall})
    per_query_table = per_query_table.round(3)
    overall_table = overall_table.round(3)
    return per_query_table, overall_table


## 2. Evaluation on `validation_labels.csv`

In [18]:
validation = labels.rename(columns={"labels": "label"})
validation_per_query, validation_overall = evaluate(VALIDATION_QUERIES, validation)
display(validation_per_query, validation_overall)


,query 1,query 2
P@5,1.000,0.800
R@5,0.385,0.400
AP@5,1.000,0.950
F1@5,0.556,0.533
RR@5,1.000,1.000
NDCG@5,1.000,0.854
P@10,1.000,0.600
R@10,0.769,0.600
AP@10,1.000,0.915
F1@10,0.870,0.600


,all queries
MAP@5,0.975
MRR@5,1.000
mean NDCG@5,0.927
MAP@10,0.958
MRR@10,1.000
mean NDCG@10,0.853


## 3. Expert judgments for the test queries

In [19]:
JUDGMENTS = PART2 / "test_queries_judgments.csv"
TOP_TO_JUDGE = 20

if not JUDGMENTS.exists():
    rows = []
    for qid, query in TEST_QUERIES.items():
        ranked = search_tf_idf(query, index, tf, idf)
        top = ranked[:TOP_TO_JUDGE]
        rank = 1
        for pid, score in top:
            title = corpus.loc[pid, "title"]
            rows.append({
                "query_id": qid,
                "query": query,
                "rank": rank,
                "pid": pid,
                "title": title,
                "label": "",
            })
            rank = rank + 1
    empty_table = pd.DataFrame(rows)
    empty_table.to_csv(JUDGMENTS, index=False)
    print("Created", JUDGMENTS.name)
    print("Fill the label column with 1 (relevant) or 0 (not relevant) and run this cell again.")

judgments = pd.read_csv(JUDGMENTS)
missing = judgments["label"].isna()
n_missing = missing.sum()
n_missing = int(n_missing)

if n_missing > 0:
    print(n_missing, "documents still without a label in", JUDGMENTS.name)
else:
    judgments["label"] = judgments["label"].astype(int)
    test_per_query, test_overall = evaluate(TEST_QUERIES, judgments)
    display(test_per_query, test_overall)


,query 3,query 4,query 5,query 6,query 7
P@5,0.200,0.400,1.000,0.200,0.600
R@5,0.067,0.167,0.250,0.083,0.200
AP@5,0.500,0.417,1.000,1.000,1.000
F1@5,0.100,0.235,0.400,0.118,0.300
RR@5,0.500,0.333,1.000,1.000,1.000
NDCG@5,0.214,0.316,1.000,0.339,0.723
P@10,0.600,0.700,1.000,0.200,0.800
R@10,0.400,0.583,0.500,0.167,0.533
AP@10,0.486,0.557,1.000,0.600,0.839
F1@10,0.480,0.636,0.667,0.182,0.640


,all queries
MAP@5,0.783
MRR@5,0.767
mean NDCG@5,0.518
MAP@10,0.696
MRR@10,0.767
mean NDCG@10,0.630
